In [1]:
"""
================================================================
CLF1 — INPUT TYPE CLASSIFIER — TRAINING (v2)
================================================================
Model  : distilbert-base-uncased
Task   : 5-class text classification
Classes: command | general | out_of_scope |
         confirmation | multiple_command
================================================================
Install before running:
    pip install transformers torch scikit-learn
================================================================
"""

import json
import numpy as np
import torch
from pathlib import Path
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)

# ================================================
# CONFIG
# ================================================

DATASET_PATH = Path(r"/content/drive/MyDrive/DataSet/LatestProperDatasets/CLF1CompleteDataSet.json")
SAVE_DIR     = Path(r"/content/drive/MyDrive/Models/EVA/clf1")

MODEL_NAME   = "distilbert-base-uncased"
MAX_LEN      = 64
BATCH_SIZE   = 16
EPOCHS       = 10
LR           = 2e-5
VAL_SPLIT    = 0.15
SEED         = 42
EARLY_STOP   = 3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================================================
# LABEL MAP — 5 classes
# ================================================

LABELS = [
    "command",
    "general",
    "out_of_scope",
    "confirmation",
    "multiple_command",
    "help"
]

label2id = {label: idx for idx, label in enumerate(LABELS)}
id2label = {idx: label for label, idx in label2id.items()}

# ================================================
# DATASET CLASS
# ================================================

class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long),
        }

# ================================================
# LOAD DATA
# ================================================

def load_data(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Safety check — catch any leftover salutation_command
    invalid = [ex for ex in data if ex["intent"] not in label2id]
    if invalid:
        print(f"\nWARNING: {len(invalid)} examples have unknown intents:")
        unknown = set(ex["intent"] for ex in invalid)
        for u in unknown:
            print(f"  → '{u}' — remove these from dataset before training")
        raise ValueError("Dataset contains invalid intents. Fix before proceeding.")

    texts  = [ex["text"] for ex in data]
    labels = [label2id[ex["intent"]] for ex in data]

    print(f"\nDataset loaded : {len(texts)} examples")
    print("\nClass distribution:")
    dist = Counter(ex["intent"] for ex in data)
    for label in LABELS:
        count = dist.get(label, 0)
        bar   = "█" * (count // 5)
        print(f"  {label:<22} → {count:>4}  {bar}")

    return texts, labels

# ================================================
# TRAIN ONE EPOCH
# ================================================

def train_epoch(model, loader, optimizer, scheduler, loss_fn, device):
    model.train()
    total_loss = 0
    correct    = 0
    total      = 0

    for batch in loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = loss_fn(outputs.logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds       = torch.argmax(outputs.logits, dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    return total_loss / len(loader), correct / total

# ================================================
# EVALUATE ONE EPOCH
# ================================================

def eval_epoch(model, loader, loss_fn, device):
    model.eval()
    total_loss = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)

            outputs     = model(input_ids=input_ids, attention_mask=attention_mask)
            loss        = loss_fn(outputs.logits, labels)
            total_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = np.mean(np.array(all_preds) == np.array(all_labels))
    return avg_loss, acc, all_preds, all_labels

# ================================================
# MAIN
# ================================================

def main():
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    print(f"Device  : {DEVICE}")
    print(f"Model   : {MODEL_NAME}")
    print(f"Classes : {LABELS}")

    # ── Load and split ─────────────────────────
    texts, labels = load_data(DATASET_PATH)

    train_texts, val_texts, train_labels, val_labels = train_test_split(
        texts, labels,
        test_size=VAL_SPLIT,
        random_state=SEED,
        stratify=labels,
    )
    print(f"\nTrain : {len(train_texts)} | Val : {len(val_texts)}")

    # ── Tokenizer ──────────────────────────────
    tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

    train_dataset = IntentDataset(train_texts, train_labels, tokenizer, MAX_LEN)
    val_dataset   = IntentDataset(val_texts,   val_labels,   tokenizer, MAX_LEN)

    train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)

    # ── Class weights ──────────────────────────
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(len(LABELS)),
        y=train_labels,
    )
    weight_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
    loss_fn       = torch.nn.CrossEntropyLoss(weight=weight_tensor)

    print(f"\nClass weights:")
    for label, w in zip(LABELS, class_weights):
        print(f"  {label:<22} → {w:.3f}")

    # ── Model ──────────────────────────────────
    model = DistilBertForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABELS),
        id2label=id2label,
        label2id=label2id,
    )
    model.to(DEVICE)

    # ── Optimizer + Scheduler ──────────────────
    optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps  = len(train_loader) * EPOCHS
    warmup_steps = int(0.1 * total_steps)
    scheduler    = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    # ── Training loop ──────────────────────────
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    best_val_loss = float("inf")
    no_improve    = 0
    best_epoch    = 0

    print(f"\n{'='*60}")
    print(f"{'Epoch':<8} {'Train Loss':<14} {'Train Acc':<14} {'Val Loss':<12} {'Val Acc'}")
    print(f"{'='*60}")

    for epoch in range(1, EPOCHS + 1):

        train_loss, train_acc = train_epoch(
            model, train_loader, optimizer, scheduler, loss_fn, DEVICE
        )
        val_loss, val_acc, val_preds, val_true = eval_epoch(
            model, val_loader, loss_fn, DEVICE
        )

        tag = ""
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch    = epoch
            no_improve    = 0
            tag           = " ← best"
            model.save_pretrained(SAVE_DIR)
            tokenizer.save_pretrained(SAVE_DIR)
        else:
            no_improve += 1

        print(
            f"  {epoch:<6} "
            f"{train_loss:<14.4f} "
            f"{train_acc * 100:<14.2f} "
            f"{val_loss:<12.4f} "
            f"{val_acc * 100:.2f}{tag}"
        )

        if no_improve >= EARLY_STOP:
            print(f"\nEarly stopping at epoch {epoch}. Best was epoch {best_epoch}.")
            break

    # ── Final report ───────────────────────────
    print(f"\n{'='*60}")
    print("Classification Report (Validation Set):")
    print(f"{'='*60}")
    print(classification_report(val_true, val_preds, target_names=LABELS))

    print("Confusion Matrix (rows=actual, cols=predicted):")
    cm = confusion_matrix(val_true, val_preds)
    print(f"{'':22}" + "".join(f"{l[:7]:>9}" for l in LABELS))
    for i, row in enumerate(cm):
        print(f"  {LABELS[i]:<20}" + "".join(f"{v:>9}" for v in row))

    # ── Save label map ──────────────────────────
    label_map = {
        "label2id": label2id,
        "id2label": {str(k): v for k, v in id2label.items()}
    }
    with open(SAVE_DIR / "label_map.json", "w") as f:
        json.dump(label_map, f, indent=2)

    print(f"\nModel saved to : {SAVE_DIR}")
    print(f"Best epoch     : {best_epoch}")
    print(f"Best val loss  : {best_val_loss:.4f}")


if __name__ == "__main__":
    main()

Device  : cuda
Model   : distilbert-base-uncased
Classes : ['command', 'general', 'out_of_scope', 'confirmation', 'multiple_command', 'help']

Dataset loaded : 8981 examples

Class distribution:
  command                → 6500  ████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]


Class weights:
  command                → 0.230
  general                → 4.298
  out_of_scope           → 2.000
  confirmation           → 4.643
  multiple_command       → 1.497
  help                   → 24.003


... (5 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  1      1.0700         70.63          0.4035       96.44 ← best


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  2      0.3488         97.83          0.2901       98.37 ← best
  3      0.2103         98.91          0.4208       97.85
  4      0.1011         99.44          0.3565       98.44
  5      0.0602         99.63          0.5186       98.44

Early stopping at epoch 5. Best was epoch 2.

Classification Report (Validation Set):
                  precision    recall  f1-score   support

         command       1.00      1.00      1.00       976
         general       0.83      0.94      0.88        52
    out_of_scope       0.97      0.97      0.97       112
    confirmation       0.98      0.88      0.92        49
multiple_command       1.00      1.00      1.00       150
            help       0.38      0.33      0.35         9

        accuracy                           0.98      1348
       macro avg       0.86      0.85      0.86      1348
    weighted avg       0.98      0.98      0.98      1348

Confusion Matrix (rows=actual, cols=predicted):
                        command  general  o

In [ ]:
"""
================================================================
CLF1 — INPUT TYPE CLASSIFIER — TESTING (v2)
================================================================
Run AFTER clf1_train.py has saved the model.

Two modes:
  1. Batch test  — point TEST_PATH to a held-out JSON file
  2. Interactive — type inputs manually, see predictions

Set TEST_PATH = None to skip batch eval.
================================================================
"""

import json
import numpy as np
import torch
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
)

# ================================================
# CONFIG
# ================================================

MODEL_DIR  = Path(r"/content/drive/MyDrive/Models/clf1")
TEST_PATH  = Path(r"/content/drive/MyDrive/DataSet/clf1_test.json")  # set None to skip

MAX_LEN    = 64
BATCH_SIZE = 32
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ================================================
# LOAD MODEL
# ================================================

def load_model(model_dir: Path):
    print(f"Loading model from : {model_dir}\n")

    tokenizer = DistilBertTokenizerFast.from_pretrained(model_dir)
    model     = DistilBertForSequenceClassification.from_pretrained(model_dir)
    model.to(DEVICE)
    model.eval()

    with open(model_dir / "label_map.json", "r") as f:
        label_map = json.load(f)

    id2label = {int(k): v for k, v in label_map["id2label"].items()}
    label2id = label_map["label2id"]

    print(f"Classes : {list(label2id.keys())}")
    return tokenizer, model, id2label, label2id

# ================================================
# DATASET FOR BATCH EVAL
# ================================================

class TestDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label":          torch.tensor(self.labels[idx], dtype=torch.long),
        }

# ================================================
# BATCH EVALUATION
# ================================================

def batch_evaluate(test_path, tokenizer, model, id2label, label2id):
    print(f"\n{'='*55}")
    print(f"BATCH EVALUATION : {test_path}")
    print(f"{'='*55}")

    with open(test_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Validate intents
    invalid = [ex for ex in data if ex["intent"] not in label2id]
    if invalid:
        print(f"WARNING: {len(invalid)} examples have unknown intents — skipping them.")
        data = [ex for ex in data if ex["intent"] in label2id]

    texts  = [ex["text"] for ex in data]
    labels = [label2id[ex["intent"]] for ex in data]

    print(f"Test examples : {len(texts)}\n")

    dataset = TestDataset(texts, labels, tokenizer, MAX_LEN)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    all_preds  = []
    all_labels = []
    all_confs  = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            batch_labels   = batch["label"]

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs   = torch.softmax(outputs.logits, dim=1)
            preds   = torch.argmax(probs, dim=1)
            confs   = torch.max(probs, dim=1).values

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch_labels.numpy())
            all_confs.extend(confs.cpu().numpy())

    label_names = [id2label[i] for i in range(len(id2label))]

    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    print(f"Overall Accuracy : {acc * 100:.2f}%")
    print(f"Avg Confidence   : {np.mean(all_confs) * 100:.2f}%\n")

    print("Classification Report:")
    print(classification_report(all_labels, all_preds, target_names=label_names))

    print("Confusion Matrix (rows=actual, cols=predicted):")
    cm = confusion_matrix(all_labels, all_preds)
    print(f"{'':22}" + "".join(f"{l[:7]:>9}" for l in label_names))
    for i, row in enumerate(cm):
        print(f"  {label_names[i]:<20}" + "".join(f"{v:>9}" for v in row))

    # Wrong predictions
    wrong = [
        (texts[i], id2label[all_labels[i]], id2label[all_preds[i]], all_confs[i])
        for i in range(len(texts))
        if all_preds[i] != all_labels[i]
    ]

    print(f"\n{'='*55}")
    print(f"Wrong Predictions : {len(wrong)} / {len(texts)}")
    print(f"{'='*55}")

    for text, expected, got, conf in wrong[:20]:
        print(f"  Text     : {text}")
        print(f"  Expected : {expected}")
        print(f"  Got      : {got}  (conf: {conf*100:.1f}%)")
        print()

    if not wrong:
        print("  No wrong predictions.")

# ================================================
# SINGLE PREDICTION
# ================================================

def predict(text: str, tokenizer, model, id2label):
    encoding = tokenizer(
        text,
        max_length=MAX_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    input_ids      = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = torch.softmax(outputs.logits, dim=1).squeeze(0)

    pred_id    = torch.argmax(probs).item()
    confidence = probs[pred_id].item()
    label      = id2label[pred_id]
    all_probs  = {id2label[i]: round(probs[i].item() * 100, 2) for i in range(len(id2label))}

    return label, confidence, all_probs

# ================================================
# INTERACTIVE MODE
# ================================================

def interactive_mode(tokenizer, model, id2label):
    print(f"\n{'='*55}")
    print("INTERACTIVE MODE — type input to classify")
    print("Commands: 'quit' or 'exit' to stop")
    print(f"{'='*55}\n")

    while True:
        try:
            text = input("Input: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nExiting.")
            break

        if not text:
            continue
        if text.lower() in ("quit", "exit"):
            print("Exiting.")
            break

        label, confidence, all_probs = predict(text, tokenizer, model, id2label)

        print(f"  Prediction : {label}")
        print(f"  Confidence : {confidence * 100:.2f}%")
        print(f"  All probs  :")
        for cls, prob in sorted(all_probs.items(), key=lambda x: -x[1]):
            bar = "█" * int(prob // 5)
            print(f"    {cls:<22} {prob:>6.2f}%  {bar}")
        print()

# ================================================
# MAIN
# ================================================

def main():
    print(f"Device : {DEVICE}")

    tokenizer, model, id2label, label2id = load_model(MODEL_DIR)

    if TEST_PATH is not None and TEST_PATH.exists():
        batch_evaluate(TEST_PATH, tokenizer, model, id2label, label2id)
    elif TEST_PATH is not None:
        print(f"\nTest file not found : {TEST_PATH}")
        print("Skipping batch evaluation.\n")

    interactive_mode(tokenizer, model, id2label)


if __name__ == "__main__":
    main()

Device : cuda
Loading model from : /content/drive/MyDrive/Models/clf1



Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Classes : ['command', 'general', 'out_of_scope', 'confirmation', 'multiple_command']

BATCH EVALUATION : /content/drive/MyDrive/DataSet/clf1_test.json
Test examples : 20

Overall Accuracy : 100.00%
Avg Confidence   : 88.44%

Classification Report:
                  precision    recall  f1-score   support

         command       1.00      1.00      1.00         5
         general       1.00      1.00      1.00         5
    out_of_scope       1.00      1.00      1.00         5
    confirmation       1.00      1.00      1.00         3
multiple_command       1.00      1.00      1.00         2

        accuracy                           1.00        20
       macro avg       1.00      1.00      1.00        20
    weighted avg       1.00      1.00      1.00        20

Confusion Matrix (rows=actual, cols=predicted):
                        command  general  out_of_  confirm  multipl
  command                     5        0        0        0        0
  general                     0        5   